# Evaluasi LLM insight dan smart query

Menguji grounding, schema respons, disclaimer, dan map action melalui fallback template deterministik. Sesuai kontrak, notebook ini tidak melatih model lokal dan tidak membuat .pkl.

Semua input dan output saat ini adalah prototipe sintetis. Hasil tidak boleh dianggap sebagai observasi lapangan atau rekomendasi bisnis/investasi produksi.

In [1]:
from pathlib import Path

def find_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'ml' / 'DATASET_CATALOG.md').exists():
            return candidate
    raise FileNotFoundError('Jalankan notebook dari dalam repository TCI')

ROOT = find_root()
ML_ROOT = ROOT / 'ml'
SEED = 20260911
STATUS = 'synthetic_prototype'
print(f'Project root: {ROOT}')

Project root: C:\Users\axels\Axel Documents\Documents\BINUS\Lomba\MAPID WebGIS (Top 50)\App\TCI


In [2]:

import json
import platform
from datetime import datetime, timezone

import pandas as pd

feature_dir = ML_ROOT / "llm_insight"
evaluation_path = feature_dir / "data" / "smart_query_evaluation.csv"
outputs_dir = feature_dir / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(ML_ROOT / "regression_scoring" / "outputs" / "station_scores.csv")
typologies = pd.read_csv(ML_ROOT / "clustering" / "outputs" / "station_typologies.csv")
recommendations = pd.read_csv(ML_ROOT / "business_classification" / "outputs" / "location_recommendations.csv")
forecasts = pd.read_csv(ML_ROOT / "forecasting" / "outputs" / "station_forecasts.csv")
cases = pd.read_csv(evaluation_path)

contexts = {}
for score in scores.itertuples(index=False):
    station_id = score.station_id
    typology = typologies.loc[typologies.station_id == station_id].iloc[0]
    recommendation = recommendations.loc[(recommendations.station_id == station_id) & (recommendations.buffer_radius_m == 750)].iloc[0]
    station_forecasts = forecasts.loc[forecasts.station_id == station_id]
    actual = station_forecasts.loc[station_forecasts.record_type == "actual"].tail(3)
    forecast = station_forecasts.loc[station_forecasts.record_type == "forecast"]
    contexts[station_id] = {
        "station_id": station_id,
        "station_name": score.station_name,
        "line": score.line,
        "as_of_period": score.period,
        "investment_potential_score": float(score.investment_score),
        "investment_rank": int(score.rank),
        "score_method": score.score_method,
        "typology": {"cluster_id": int(typology.cluster_id), "label": typology.typology_label},
        "passenger_volume_recent_3m": actual.actual_passengers.astype(int).tolist(),
        "forecast_next_3m": forecast[["period", "predicted_passengers", "lower_80", "upper_80", "method"]].to_dict("records"),
        "business_recommendation": {"location_id": recommendation.location_id, "category": recommendation.recommended_category, "confidence": float(recommendation.recommendation_confidence), "buffer_radius_m": int(recommendation.buffer_radius_m)},
        "source_status": STATUS,
    }
context_path = outputs_dir / "station_insight_context.jsonl"
with context_path.open("w", encoding="utf-8") as file:
    for payload in contexts.values():
        file.write(json.dumps(payload, ensure_ascii=False) + "\n")

duplicate_contexts = len(contexts) != len(scores)
missing_context_ids = sorted(set(cases.station_id) - set(contexts))
invalid_status = int((cases.source_status != STATUS).sum()) + sum(value.get("source_status") != STATUS for value in contexts.values()) + int((scores.source_status != STATUS).sum()) + int((typologies.source_status != STATUS).sum()) + int((recommendations.source_status != STATUS).sum()) + int((forecasts.source_status != STATUS).sum())
quality_passed = not (duplicate_contexts or missing_context_ids or invalid_status)
quality_report = {"passed": quality_passed, "context_count": len(contexts), "evaluation_case_count": len(cases), "duplicate_context_station_ids": duplicate_contexts, "missing_context_station_ids": missing_context_ids, "invalid_source_status_count": invalid_status}
(outputs_dir / "data_quality_report.json").write_text(json.dumps(quality_report, indent=2), encoding="utf-8")
if not quality_passed:
    raise ValueError("LLM insight quality gate failed; responses are unavailable")

def nested_value(payload, dotted_key):
    value = payload
    for key in dotted_key.split("."):
        value = value[key]
    return value

def deterministic_response(context, intent):
    station = context["station_name"]
    limitation = "Semua angka dan atribut adalah prototipe sintetis, bukan observasi lapangan atau dasar keputusan final."
    evidence = [{"key": "station_name", "value": station}, {"key": "source_status", "value": context["source_status"]}]
    if intent == "station_summary":
        score = context["investment_potential_score"]
        summary = f"{station} memiliki skor potensi sintetis {score:.2f}/100 pada {context['as_of_period']}."
        evidence.append({"key": "investment_potential_score", "value": score})
        map_action = {"type": "focus_station", "station_id": context["station_id"]}
    elif intent == "business_category":
        recommendation = context["business_recommendation"]
        summary = f"Kategori yang dapat dieksplorasi pada radius {recommendation['buffer_radius_m']} m di {station} adalah {recommendation['category']}."
        evidence.append({"key": "business_recommendation.category", "value": recommendation["category"]})
        map_action = None
    else:
        summary = f"Insight {station} belum dapat dianggap andal untuk keputusan nyata karena seluruh input masih sintetis."
        map_action = None
    response = {"summary": summary, "evidence": evidence, "limitations": [limitation]}
    if map_action and map_action["station_id"] in contexts:
        response["map_action"] = map_action
    return response

results, passed_count = [], 0
for case in cases.itertuples(index=False):
    context = contexts[case.station_id]
    response = deterministic_response(context, case.intent)
    required_keys = json.loads(case.required_fact_keys)
    evidence_by_key = {item["key"]: item["value"] for item in response["evidence"]}
    grounded = all(key in evidence_by_key and evidence_by_key[key] == nested_value(context, key) for key in required_keys)
    limitation_passed = (not bool(case.must_include_limitation)) or any("sintetis" in text.lower() for text in response["limitations"])
    forbidden_advice_passed = not any(term in response["summary"].lower() for term in ["pasti untung", "wajib investasi", "izin dijamin"])
    map_action_valid = "map_action" not in response or response["map_action"].get("station_id") in contexts
    passed = grounded and limitation_passed and forbidden_advice_passed and map_action_valid
    passed_count += int(passed)
    results.append({"case_id": case.case_id, "station_id": case.station_id, "intent": case.intent, "response": response, "checks": {"grounded_required_facts": grounded, "includes_synthetic_limitation": limitation_passed, "no_forbidden_final_advice": forbidden_advice_passed, "valid_map_action": map_action_valid}, "passed": passed, "source_status": STATUS})

with (outputs_dir / "insight_evaluations.jsonl").open("w", encoding="utf-8") as file:
    for result in results:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")
metrics = {"cases": len(results), "passed": passed_count, "pass_rate": passed_count / len(results), "mode": "deterministic_template_fallback", "source_status": STATUS, "limitation": "This evaluates grounding and response schema, not LLM model quality or fine-tuning."}
(outputs_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
schema = {"required_response_fields": ["summary", "evidence", "limitations"], "optional_response_fields": ["map_action"], "allowed_map_action": {"type": "focus_station", "station_id": "must exist in validated context"}, "local_model_artifact": None, "reason_no_pkl": "LLM API is not a local model trained by this project."}
(outputs_dir / "feature_schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
manifest = {"run_at_utc": datetime.now(timezone.utc).isoformat(), "sources": ["ml/regression_scoring/outputs/station_scores.csv", "ml/clustering/outputs/station_typologies.csv", "ml/business_classification/outputs/location_recommendations.csv", "ml/forecasting/outputs/station_forecasts.csv", str(evaluation_path.relative_to(ROOT))], "context_output": str(context_path.relative_to(ROOT)), "records": {"contexts": len(contexts), "cases": len(cases)}, "python": platform.python_version(), "source_status": STATUS, "limitations": "Prompt/guardrail fixture only; no local model and intentionally no .pkl artifact."}
(outputs_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({"quality_passed": quality_passed, "evaluation_pass_rate": metrics["pass_rate"], "output": str(outputs_dir / 'insight_evaluations.jsonl'), "pkl_created": False}, indent=2))


{
  "quality_passed": true,
  "evaluation_pass_rate": 1.0,
  "output": "C:\\Users\\axels\\Axel Documents\\Documents\\BINUS\\Lomba\\MAPID WebGIS (Top 50)\\App\\TCI\\ml\\llm_insight\\outputs\\insight_evaluations.jsonl",
  "pkl_created": false
}
